# TuneDinoV2 Stage 1 Kaggle Notebook (Finetune DINOv2)

Notebook này chạy Stage 1 trên Kaggle GPU theo hướng finetune DINOv2:

- nạp decoder ImageNet DINOv2 làm khởi tạo;
- cho phép gắn thêm `stage1_ckpt_path` nếu bạn đã có checkpoint Stage 1 cũ;
- mở train encoder DINOv2 với `encoder_lr` nhỏ hơn decoder;
- log đầy đủ `L1`, `LPIPS`, GAN losses, latent stats, learning rate, grad norm, ảnh reconstruction lên W&B project `TuneDinoV2`.

Notebook hỗ trợ cả `CelebA` và `CelebA-HQ` bằng biến `dataset_name` ở cell cấu hình.


In [ ]:
%cd /kaggle/working
!rm -rf RAE
!git clone https://github.com/sontungkieu/RAE
%cd /kaggle/working/RAE

import os
import shutil
import subprocess
from pathlib import Path

subprocess.run(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)

uv_bin = shutil.which("uv")
if uv_bin is None:
    for candidate in (
        Path.home() / ".local/bin/uv",
        Path("/root/.local/bin/uv"),
        Path("/usr/local/bin/uv"),
    ):
        if candidate.exists():
            uv_bin = candidate.as_posix()
            break

if uv_bin is None:
    raise FileNotFoundError("uv install finished but the uv binary is still missing from PATH and ~/.local/bin")

os.environ["UV_BIN"] = uv_bin
os.environ["PATH"] = f"{Path(uv_bin).parent}:{os.environ.get('PATH', '')}"
print("Using uv binary:", uv_bin)


In [ ]:
import os
import subprocess

os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/.venv"
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"

uv_bin = os.environ["UV_BIN"]
subprocess.run([uv_bin, "sync", "-q"], check=True, cwd="/kaggle/working/RAE")
subprocess.run(["nvidia-smi"], check=False)
print("Repo dependencies are synced into /tmp/.venv")


In [ ]:
import os
import pathlib

PROJECT = "TuneDinoV2"
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "TungBangDSLab")

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    wandb_token = None
    for secret_name in ("WANDB_Tung", "WANDB2", "WANDB_KEY"):
        try:
            wandb_token = secrets.get_secret(secret_name)
            print(f"Loaded W&B secret from {secret_name}")
            break
        except Exception:
            continue
    if wandb_token is None:
        raise RuntimeError("No W&B Kaggle secret found")

    os.environ["WANDB_API_KEY"] = wandb_token
    os.environ["WANDB_KEY"] = wandb_token
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
    os.environ["PROJECT"] = PROJECT
    netrc = pathlib.Path.home() / ".netrc"
    netrc.write_text(f"machine api.wandb.ai login user password {wandb_token}\n", encoding="utf-8")
    os.chmod(netrc, 0o600)
except Exception as exc:
    print(f"Skipping Kaggle W&B bootstrap: {exc}")

print("W&B entity:", WANDB_ENTITY)
print("W&B project:", PROJECT)


In [ ]:
import json
import textwrap
from datetime import datetime
from pathlib import Path

repo_root = Path("/kaggle/working/RAE")
dataset_name = "celebahq"  # change to "celeba" if needed
prepare_dataset = True
celeb_hq_dataset_id = "eurecom-ds/celeba-hq-256"
celeba_src_dir = Path("/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba")
celeba_split_csv = Path("/kaggle/input/datasets/jessicali9530/celeba-dataset/list_eval_partition.csv")
face_root = Path("/kaggle/working/celebahq256_imgfolder" if dataset_name == "celebahq" else "/kaggle/working/celeba256_imgfolder")
train_root = face_root / "train"
val_root = face_root / "val"
results_dir = Path("/kaggle/working/results_stage1")
generated_cfg_dir = repo_root / "configs" / "stage1" / "training" / "generated"
generated_cfg_dir.mkdir(parents=True, exist_ok=True)
config_path = generated_cfg_dir / "finetune_tunedinov2_stage1.yaml"
run_timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
exp_prefix = "TuneDinoV2-stage1-finetune-dino"
wandb_group = "stage1-finetune-dino"
wandb_tags = "stage1,mode:finetune,encoder:trainable,decoder:imagenet_init,project:TuneDinoV2"
stage1_ckpt_path = None  # set to an existing Stage 1 checkpoint to continue from your best run

stage1_ckpt_yaml = "null" if stage1_ckpt_path is None else f"'{stage1_ckpt_path}'"
config_text = textwrap.dedent(
    f"""
    stage_1:
      target: stage1.RAE
      ckpt: {stage1_ckpt_yaml}
      params:
        encoder_cls: 'Dinov2withNorm'
        encoder_config_path: 'facebook/dinov2-with-registers-base'
        encoder_input_size: 224
        encoder_params:
          dinov2_path: 'facebook/dinov2-with-registers-base'
          normalize: true
        decoder_config_path: 'configs/decoder/ViTXL'
        pretrained_decoder_path: 'models/decoders/dinov2/wReg_base/ViTXL_n08/model.pt'
        noise_tau: 0.0
        reshape_to_2d: true

    training:
      global_seed: 0
      epochs: 12
      batch_size: 8
      num_workers: 4
      image_size: 256
      precision: fp16
      log_every: 20
      image_log_every: 100
      eval_every: 1
      save_every: 1
      recon_weight: 1.0
      train_encoder: true
      encoder_lr: 1.0e-5
      random_flip: true
      clip_grad: 1.0
      num_visuals: 8
      optimizer:
        lr: 1.0e-4
        betas: [0.9, 0.95]
        weight_decay: 0.0
      scheduler:
        type: cosine
        warmup_epochs: 1
        decay_end_epoch: 12
        base_lr: 1.0e-4
        final_lr: 2.0e-5

    gan:
      disc:
        arch:
          dino_ckpt_path: 'models/discs/dino_vit_small_patch8_224.pth'
          ks: 9
          norm_type: 'bn'
          using_spec_norm: true
          recipe: 'S_8'
        optimizer:
          lr: 2.0e-4
          betas: [0.5, 0.9]
          weight_decay: 0.0
        scheduler:
          type: cosine
          warmup_epochs: 1
          decay_end_epoch: 12
          base_lr: 2.0e-4
          final_lr: 2.0e-5
        augment:
          prob: 1.0
          cutout: 0.0
      loss:
        disc_loss: hinge
        gen_loss: vanilla
        disc_weight: 0.75
        perceptual_weight: 1.0
        disc_start: 1
        disc_upd_start: 1
        lpips_start: 0
        max_d_weight: 10000.0
        disc_updates: 1

    data:
      train_path: '{train_root.as_posix()}'

    eval:
      data_path: '{val_root.as_posix()}'
      batch_size: 8
      num_workers: 2
      max_batches: 50
    """
).strip() + "\n"
config_path.write_text(config_text, encoding="utf-8")

run_info = {
    "title": "TuneDinoV2 Stage 1 finetune DINOv2",
    "dataset_name": dataset_name,
    "train_root": train_root.as_posix(),
    "val_root": val_root.as_posix(),
    "config_path": config_path.as_posix(),
    "results_dir": results_dir.as_posix(),
    "wandb_group": wandb_group,
    "wandb_tags": wandb_tags,
    "exp_name": f"{exp_prefix}-{dataset_name}-{run_timestamp}",
}
print(json.dumps(run_info, indent=2))
print(config_text)


In [ ]:
import os
import subprocess

UV_BIN = os.environ["UV_BIN"]


subprocess.run(
    [
        UV_BIN,
        "run",
        "hf",
        "download",
        "nyu-visionx/RAE-collections",
        "discs/dino_vit_small_patch8_224.pth",
        "--local-dir",
        "models",
    ],
    check=True,
    cwd=repo_root,
)

subprocess.run(
    [
        UV_BIN,
        "run",
        "hf",
        "download",
        "nyu-visionx/RAE-collections",
        "decoders/dinov2/wReg_base/ViTXL_n08/model.pt",
        "--local-dir",
        "models",
    ],
    check=True,
    cwd=repo_root,
)

print("Model assets are ready under", repo_root / "models")


In [ ]:
import csv
import os
import subprocess
from PIL import Image

UV_BIN = os.environ["UV_BIN"]



def center_crop_resize_256(src_path: Path, dst_path: Path) -> None:
    with Image.open(src_path) as image:
        image = image.convert("RGB")
        width, height = image.size
        crop = min(width, height)
        left = (width - crop) // 2
        top = (height - crop) // 2
        image = image.crop((left, top, left + crop, top + crop))
        image = image.resize((256, 256), Image.BICUBIC)
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        image.save(dst_path, quality=95)


if prepare_dataset and dataset_name == "celebahq":
    if not face_root.exists():
        subprocess.run(
            [
                UV_BIN,
                "run",
                "python",
                "src_jax/export_celebahq_hf.py",
                "--dataset",
                celeb_hq_dataset_id,
                "--output",
                face_root.as_posix(),
            ],
            check=True,
            cwd=repo_root,
        )
    else:
        print("CelebA-HQ ImageFolder already exists:", face_root)
elif prepare_dataset and dataset_name == "celeba":
    if not celeba_src_dir.exists():
        raise FileNotFoundError(f"Missing CelebA source directory: {celeba_src_dir}")
    if not celeba_split_csv.exists():
        raise FileNotFoundError(f"Missing CelebA split csv: {celeba_split_csv}")
    if not face_root.exists():
        split_map = {"0": "train", "1": "val", "2": "test"}
        created = 0
        with celeba_split_csv.open("r", encoding="utf-8") as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                split = split_map[row["partition"]]
                src = celeba_src_dir / row["image_id"]
                dst = face_root / split / "face" / row["image_id"]
                center_crop_resize_256(src, dst)
                created += 1
                if created % 10000 == 0:
                    print(f"prepared {created} CelebA images...")
        print(f"Prepared {created} CelebA images under {face_root}")
    else:
        print("CelebA ImageFolder already exists:", face_root)
else:
    print("Skipping dataset preparation; expecting ImageFolder roots to already exist.")

print("train root:", train_root)
print("val root:", val_root)
print("train images:", sum(1 for _ in train_root.rglob('*.jpg')) + sum(1 for _ in train_root.rglob('*.png')))
print("val images:", sum(1 for _ in val_root.rglob('*.jpg')) + sum(1 for _ in val_root.rglob('*.png')))


In [ ]:
import json
import os
import subprocess

UV_BIN = os.environ["UV_BIN"]

env = os.environ.copy()
env["WANDB_ENTITY"] = WANDB_ENTITY
env["PROJECT"] = PROJECT

cmd = [
    UV_BIN,
    "run",
    "python",
    "src/train_stage1_rae.py",
    "--config",
    config_path.as_posix(),
    "--results-dir",
    results_dir.as_posix(),
    "--exp-name",
    run_info["exp_name"],
    "--wandb",
    "--wandb-entity",
    WANDB_ENTITY,
    "--wandb-project",
    PROJECT,
    "--wandb-group",
    wandb_group,
    "--wandb-tags",
    wandb_tags,
]
subprocess.run(cmd, check=True, cwd=repo_root, env=env)

workdir = results_dir / run_info["exp_name"]
run_info["workdir"] = workdir.as_posix()
run_info["last_ckpt"] = (workdir / "checkpoints" / "last.pt").as_posix()
run_info["best_ckpt"] = (workdir / "checkpoints" / "best.pt").as_posix()
summary_path = Path("/kaggle/working") / f"{run_info['exp_name']}_summary.json"
summary_path.write_text(json.dumps(run_info, indent=2), encoding="utf-8")
print("Saved run summary to", summary_path)
print(json.dumps(run_info, indent=2))


In [ ]:
import json
from pathlib import Path

summary_candidates = sorted(Path("/kaggle/working").glob("TuneDinoV2-stage1-*_summary.json"))
if not summary_candidates:
    raise FileNotFoundError("No run summary json found under /kaggle/working")

summary_path = summary_candidates[-1]
run_info = json.loads(summary_path.read_text(encoding="utf-8"))
workdir = Path(run_info["workdir"])

print("Summary file:", summary_path)
print(json.dumps(run_info, indent=2))
print("\nCheckpoint directory:")
for path in sorted((workdir / "checkpoints").glob("*")):
    print("-", path.name)
